## Launching model on dataset and collecting output

### Imports

In [1]:
from datasets import load_dataset
from tqdm import tqdm

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Any, List

import torch
import sys

sys.path.append("../../../services/")
from index import Index
from process_funcs import retrieve_answer_token_index
from metric_funcs import calculate_entropy, calculate_norm_entropy

torch.random.manual_seed(42)

/workspace/Diplom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Preparation

In [2]:
ITERATIONS_COUNT = 12000
TOPK = 30
LOWER_PROB_LIMIT = 1e-8
LOWER_LOGIT_LIMIT = -1000
UPPER_LOGIT_LIMIT = 1000

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)
print(torch.cuda.get_device_properties(device))

cuda:0
_CudaDeviceProperties(name='NVIDIA L40', major=8, minor=9, total_memory=45385MB, multi_processor_count=142, uuid=441e6008-cd27-34b5-c81a-17dad9d4a894, L2_cache_size=96MB)


In [4]:
dataset = load_dataset("TIGER-Lab/MMLU-Pro", split="test")

Generating validation split: 100%|██████████| 70/70 [00:00<00:00, 14025.76 examples/s]


In [5]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    attn_implementation="eager",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = model.to(device)
model.eval()

Fetching 4 files: 100%|██████████| 4/4 [01:55<00:00, 28.91s/it] 
Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 64.13it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [6]:
class LLMInterface(Runnable):    
    def __init__(self, model, tokenizer, device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')):
        self.model = model
        self.tokenizer = tokenizer
        self.hook_handles = []
        self.device = device

        self.model = self.model.to(device)

    def invoke(        
        self,
        messages: List[Any],
        config: RunnableConfig | None = None,
        **kwargs: Any,
        ):
        hf_messages = [
            {"role": "system", "content": messages.messages[0].content},
            {"role": "user", "content": messages.messages[1].content},
        ]
                
        inputs = self.tokenizer.apply_chat_template(
            hf_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer(
            inputs,
            return_tensors="pt"
        ).to(self.model.device)
        
        
        self.model.eval()
        with torch.no_grad():
            response = self.model.generate(
                **model_inputs,
                max_new_tokens=512,
                output_scores=True,
                output_attentions=True,
                return_dict_in_generate=True,
                do_sample=True,
                temperature=0.7,
                repetition_penalty=1.05,
                top_p=0.8,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        token_data = []
        generated_ids = response.sequences[:, model_inputs["input_ids"].shape[1]:].squeeze(0)
        for i, logits in enumerate(response.scores):
            logits = logits.squeeze(0)
            probs = torch.softmax(logits, dim=-1) 
            
            top_logits, top_tok_ids = torch.topk(logits, TOPK, dim=-1)
            top_logits = torch.clamp(top_logits, min=LOWER_LOGIT_LIMIT, max=UPPER_LOGIT_LIMIT)
            
            top_probs, _ = torch.topk(probs, TOPK, dim=-1)
            top_probs = top_probs.cpu()
            
            top_tokens = []
            for j in range(top_tok_ids.shape[-1]):
                top_tokens.append(self.tokenizer.decode(top_tok_ids[j].item()))
                
            token_id = generated_ids[i]
            token_data.append({
                "token": self.tokenizer.decode(token_id),
                "prob": probs[token_id].cpu().item(),
                "logit": logits[token_id].cpu().item(),
                "top_tokens": top_tokens,
                "top_logits": top_logits,
                "top_probs": top_probs,
            })   

        output_attetntions = [
            torch.stack(
                token_attention
            ).squeeze(1).clamp(min=LOWER_PROB_LIMIT).cpu()
            for token_attention in response.attentions[1:]
        ]

        attn_entropy = [
            calculate_entropy(x)
            for x in output_attetntions
        ]
        
        norm_attn_entropy = [
            calculate_norm_entropy(x)
            for x in output_attetntions
        ]
        
        return {
            "input_text": self.tokenizer.decode(response.sequences[:, :model_inputs["input_ids"].shape[1]].cpu()[0], skip_special_tokens=True),
            "output_text": self.tokenizer.decode(response.sequences[:, model_inputs["input_ids"].shape[1]:].cpu()[0], skip_special_tokens=True),
            "score_data": token_data,
            "attention_entropy": attn_entropy,
            "norm_attention_entropy": norm_attn_entropy
        }   

In [7]:
chat_llm = LLMInterface(
    model=model,
    tokenizer=tokenizer,
    device=device
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert in answering multiple choice questions. 
            You will be given a question and options. Follow these rules strictly:

            1. Read the question and given options.
            2. Begin a short reasoning process before giving answer and surround your reasoning with <think> and </think> tags.
            3. After the chain of thoughts is complete, select the correct option number between 0 and 9.

            Follow this output format:
            <think>
            SHORT_CHAIN_OF_THOUGHTS_TEXT
            </think>
            Answer: OPTION_NUMBER
            
            Example:
            <think>
            The capital of France is Paris, which is option 2.
            </think>
            Answer: 2 

            Now answer the following question following output format.
            """,
        ),
        (
            "human",
            """
            Question: 
            {input_question}
            
            Options:
            {input_options}
            """
        )
    ]
)

llm_chain = prompt | chat_llm

In [8]:
len(dataset)

12032

### Running model on dataset and collecting results


In [9]:
def filter_func(model_response, right_answer):
    if "<think>" not in model_response["output_text"] \
        or "</think>" not in model_response["output_text"]:
        return False
    
    token_data = model_response["score_data"][
        retrieve_answer_token_index(
            model_response["score_data"]
        )
    ]
    return right_answer in token_data["top_tokens"]

In [10]:
# launching model

ERROR_LIMIT = 500

index = Index(f"../../../index_data/qwen2.5-7B_MMLU-PRO_reasoning1_{ITERATIONS_COUNT}")
index.clear()

correct_count = 0
progress_bar = tqdm(
    enumerate(dataset.select(range(ITERATIONS_COUNT))),
    total=ITERATIONS_COUNT,
    desc="Scoring MMLU-PRO",
)

for iter, data in progress_bar:
    for error_count in range(ERROR_LIMIT + 1):
        try:
            response = llm_chain.invoke(
                {
                    "input_question": dataset[iter]["question"],
                    "input_options": [
                        f"{i}:  {x}"
                        for i, x in enumerate(dataset[iter]["options"])
                    ]
                }
            )
            
            response["dataset_elem"] = dataset[iter]

            if filter_func(response, str(data["answer_index"])):
                if response["output_text"][-1] == str(
                    data["answer_index"]
                ):
                    correct_count += 1

                progress_bar.set_postfix(
                    {"Current accuracy": str(correct_count / (iter + 1))}
                )
                
                response = {
                    "iteration": iter,
                    **response
                }

                index.save_data(response, iter, logging=False)
                break
            
            if error_count == ERROR_LIMIT:
                raise Exception("Error limit exceeded")
            
        except Exception as e:
            if (error_count + 1) % 500 == 0:
                print(f"Error on iteration {iter + 1}\nError count: {error_count}\nMessage: {e}")
            if error_count == ERROR_LIMIT:
                print("Error limit exceeded")
                

Файл не существует: ../../../index_data/qwen2.5-7B_MMLU-PRO_reasoning1_12000_index.pkl
Файл не существует: ../../../index_data/qwen2.5-7B_MMLU-PRO_reasoning1_12000_data.pkl
Index is cleared successfully.


Scoring MMLU-PRO: 100%|██████████| 12000/12000 [22:57:17<00:00,  6.89s/it, Current accuracy=0.38616666666666666]    


In [11]:
index = Index(f"../../index_data/qwen2.5-7B_MMLU-PRO_reasoning1_{ITERATIONS_COUNT}")

In [12]:
print(len(index))

0
